# ARC-AGI-2 — two-pass solver

Built on the NVARC 2025 ARChitects pipeline: per-task LoRA test-time
training on `qwen3_4b_grids15_sft139`, threshold DFS decoding, and
augmentation re-scoring of the candidates.

What is different here:

- **Two passes.** Pass 1 sweeps every task on a cheap budget. The tasks
  whose results look starved or contested are re-run in pass 2 with more
  test-time training, more colour permutations and a looser decode
  threshold. Both passes feed one ranking, so a revisited task
  accumulates evidence rather than replacing it.
- **No unsloth requirement.** It is used when present, and the run
  falls back to plain transformers + peft when it is not, so the
  notebook starts on any Kaggle image with only the competition and
  the model attached.
- **Shape prior.** When every demonstration pair agrees on how the output
  shape follows from the input shape, candidates matching that shape are
  promoted ahead of the ones that do not. It reorders, never drops.

Set `BASELINE = True` in the config cell to fall back to the single-pass
public baseline.

Generated by `build_notebook.py` — edit `src/`, not this file.


In [ ]:
import time

global_end_time = time.time() + 12 * 3600 - 1800


In [ ]:
!pip uninstall -y tensorflow


In [ ]:
%%writefile preflight.py
"""Fail fast on what actually blocks a run, and only warn about the rest.

The solver imports its model backend inside spawned worker processes, so a
missing input surfaces as a ProcessRaisedException several minutes in with the
real cause buried in a spawn traceback. These checks run in seconds, before
anything is spawned.

unsloth is deliberately *not* required. It gives a faster path when present;
when it is absent the solver runs on plain transformers + peft.
"""

import importlib.util
import os
import sys

UNSLOTH_PYTHON = (3, 11)
DOCKER_IMAGE_VERSION_ID = 31090


def _importable(name, find_spec=None):
    find_spec = find_spec or importlib.util.find_spec
    try:
        return find_spec(name) is not None
    except (ImportError, ValueError):
        return False


def describe_inputs(root="/kaggle/input"):
    """What is actually mounted, so a missing input is obvious at a glance."""
    if not os.path.isdir(root):
        return [f"{root} does not exist — no inputs are attached at all"]
    entries = sorted(os.listdir(root))
    if not entries:
        return [f"{root} is empty — no inputs are attached"]
    return [f"{root}/{e}" for e in entries]


def find_problems(required_paths, find_spec=None, required=("torch", "transformers", "peft", "datasets")):
    """Blocking problems only. Empty list means the run can start."""
    problems = []

    for name in required:
        if not _importable(name, find_spec):
            problems.append(f"'{name}' is not importable; it should be in the Kaggle image.")

    for path in required_paths:
        if not os.path.exists(path):
            problems.append(
                f"missing input: {path}\n"
                f"      currently attached: " + ", ".join(describe_inputs()) + "\n"
                f"      add it under Add Input in the notebook editor "
                f"(models may need a licence accepted first)."
            )

    return problems


def find_model_problem(config):
    """Resolve the checkpoint, or explain why we cannot."""
    path = config.resolve_model_path()
    if path is not None:
        return None, path
    return (
        "no model checkpoint found under /kaggle/input.\n"
        "      expected: " + config.MODEL_PATH + "\n"
        "      currently attached: " + ", ".join(describe_inputs()) + "\n"
        "      add the model '" + config.MODEL_OWNER + "/" + config.MODEL_SLUG + "' "
        "(transformers / bfloat16) under Add Input."
    ), None


def find_warnings(find_spec=None, python=None):
    """Things worth knowing that do not stop the run."""
    python = python or sys.version_info[:2]
    warnings = []

    has_unsloth = _importable("unsloth", find_spec)

    if not has_unsloth:
        warnings.append(
            "unsloth not found — using the transformers backend. Same method, "
            "slower per task. To get the fast path, fork a notebook pinned to "
            f"Kaggle image {DOCKER_IMAGE_VERSION_ID} with the "
            "sorokin/pip-install-unsloth-flash-patch input attached."
        )
    elif python != UNSLOTH_PYTHON:
        want = ".".join(map(str, UNSLOTH_PYTHON))
        got = ".".join(map(str, python))
        warnings.append(
            f"unsloth is present but this is Python {got}, not {want}. If it "
            f"fails to import, the solver falls back to transformers."
        )

    return warnings


def check(required_paths):
    import arc_backend
    import arc_config

    print(f"python {sys.version.split()[0]}")
    for name in ("torch", "transformers", "peft", "datasets", "unsloth", "flash_attn"):
        print(f"  {name:12s} {'ok' if _importable(name) else 'missing'}")

    try:
        import torch
        print(f"  cuda devices {torch.cuda.device_count()}")
    except Exception as e:
        print(f"  cuda devices unknown ({e})")

    print("attached inputs:")
    for line in describe_inputs():
        print(f"  {line}")

    print(f"*** backend: {arc_backend.name()}")

    for w in find_warnings():
        print(f"*** note: {w}")

    problems = find_problems(required_paths)

    model_problem, model_path = find_model_problem(arc_config)
    if model_problem:
        problems.append(model_problem)
    else:
        print(f"*** model: {model_path}")
        if model_path != arc_config.MODEL_PATH:
            print(f"*** note: resolved by search; MODEL_PATH says {arc_config.MODEL_PATH}")

    if problems:
        raise RuntimeError(
            "Cannot start:\n  - " + "\n  - ".join(problems)
        )
    print("*** preflight ok")


In [ ]:
%%writefile arc_backend.py
"""Model backend, so the notebook does not depend on unsloth being present.

unsloth is not part of the Kaggle image. It comes from a utility script pinned
to a Python 3.11 image, and attaching that script to a notebook on a newer
image makes the session fail to start at all. That single dependency has been
the only thing standing between us and a run, so it is now optional:

- unsloth importable  -> fast path, byte-for-byte the behaviour of the public
  baseline (patched attention, unsloth's trainer).
- unsloth missing     -> plain transformers + peft. Same model, same LoRA
  config, same schedule, same decoding. Slower per task, which the two-pass
  budget absorbs by revisiting fewer tasks; nothing about the method changes.

Everything the solver needs from a backend goes through this module.
"""

import importlib.util
import torch

import arc_config


def _importable(name):
    try:
        return importlib.util.find_spec(name) is not None
    except (ImportError, ValueError):
        return False


HAS_UNSLOTH = _importable("unsloth")

LORA = dict(
    r=arc_config.LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj",
                    "embed_tokens", "lm_head"],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_rslora=True,
)


def name():
    return "unsloth" if HAS_UNSLOTH else "transformers"


def load_model(model_path, max_seq_length, compute_dtype, pad_token_id):
    if HAS_UNSLOTH:
        from unsloth import FastLanguageModel
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_path,
            full_finetuning=False,
            load_in_4bit=False,
            local_files_only=True,
            use_gradient_checkpointing=False,
            max_seq_length=max_seq_length,
        )
    else:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
        try:
            model = AutoModelForCausalLM.from_pretrained(
                model_path, dtype=compute_dtype, local_files_only=True,
                attn_implementation="sdpa",
            )
        except TypeError:
            model = AutoModelForCausalLM.from_pretrained(
                model_path, torch_dtype=compute_dtype, local_files_only=True,
                attn_implementation="sdpa",
            )
        model = model.cuda()

    if tokenizer.pad_token_id != pad_token_id:
        print(f"*** pad_token_id {tokenizer.pad_token_id} -> {pad_token_id}")
        tokenizer.pad_token_id = pad_token_id
    return model, tokenizer


def neutralise_optional_backend_probes():
    """Stop peft's quantiser probes from raising on a plain LoRA injection.

    peft's LoRA dispatcher asks each optional backend whether it is available.
    `is_torchao_available` does not answer False when the installed torchao is
    older than peft wants — it raises ImportError. The Kaggle image ships
    torchao 0.10 against a peft that wants >0.16, so injecting an ordinary
    bf16 LoRA adapter dies on a quantiser we never use. Answer False for it.
    """
    patched = []
    try:
        from peft import import_utils
    except ImportError:
        return patched

    probe = getattr(import_utils, "is_torchao_available", None)
    if probe is None:
        return patched
    try:
        probe()
        return patched
    except ImportError:
        pass

    false = lambda: False
    import_utils.is_torchao_available = false
    patched.append("peft.import_utils")

    try:
        from peft.tuners.lora import torchao as lora_torchao
    except ImportError:
        return patched
    if hasattr(lora_torchao, "is_torchao_available"):
        lora_torchao.is_torchao_available = false
        patched.append("peft.tuners.lora.torchao")
    return patched


def get_peft_model(model):
    if HAS_UNSLOTH:
        from unsloth import FastLanguageModel
        return FastLanguageModel.get_peft_model(
            model, use_gradient_checkpointing=False, random_state=42,
            loftq_config=None, **LORA,
        )
    patched = neutralise_optional_backend_probes()
    if patched:
        print(f"*** disabled torchao probe in: {', '.join(patched)}")
    from peft import LoraConfig, get_peft_model as peft_get_peft_model
    return peft_get_peft_model(model, LoraConfig(task_type="CAUSAL_LM", **LORA))


def for_training(model):
    if HAS_UNSLOTH:
        from unsloth import FastLanguageModel
        return FastLanguageModel.for_training(model)
    model.train()
    return model


def for_inference(model):
    if HAS_UNSLOTH:
        from unsloth import FastLanguageModel
        return FastLanguageModel.for_inference(model)

    if hasattr(model, "gradient_checkpointing_disable"):
        model.gradient_checkpointing_disable()
    config = getattr(model, "config", None)
    if config is not None and hasattr(config, "use_cache"):
        config.use_cache = True

    model.eval()
    return model


def eval_strategy_key():
    """`evaluation_strategy` was renamed to `eval_strategy` in transformers
    4.46, and passing the name this version does not know raises."""
    try:
        import inspect
        from transformers import TrainingArguments
        fields = inspect.signature(TrainingArguments.__init__).parameters
    except Exception:
        return "eval_strategy"
    return "eval_strategy" if "eval_strategy" in fields else "evaluation_strategy"


def adjust_train_args(train_args):
    """Memory settings the plain path needs and the unsloth path does not.

    The reference runs without gradient checkpointing because unsloth's fused
    kernels keep the activations small enough. Plain transformers does not:
    thirty-six layers of an 8192-token sequence will not fit beside a rank-256
    adapter and its optimiser state on a 22 GB L4. Recomputing activations in
    the backward pass costs perhaps a third of the speed and changes nothing
    about the result.
    """
    out = dict(train_args)
    if HAS_UNSLOTH:
        return out
    out["gradient_checkpointing"] = True
    out["gradient_checkpointing_kwargs"] = {"use_reentrant": False}
    return out


def build_trainer(model, tokenizer, collator, samples, train_args,
                  max_seq_length, fixed_trainer_cls=None, callbacks=None):
    """Trainer over `samples`, a list of dicts carrying a 'text' field."""
    from datasets import Dataset

    if HAS_UNSLOTH:
        from unsloth import UnslothTrainingArguments
        return fixed_trainer_cls(
            model=model,
            tokenizer=tokenizer,
            data_collator=collator,
            train_dataset=Dataset.from_list(samples),
            dataset_text_field="text",
            max_seq_length=max_seq_length,
            args=UnslothTrainingArguments(**train_args),
            callbacks=callbacks,
        )

    from transformers import Trainer, TrainingArguments

    dataset = Dataset.from_list(samples)
    columns = dataset.column_names

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=max_seq_length)

    dataset = dataset.map(tokenize, batched=True, remove_columns=columns)

    train_args = adjust_train_args(train_args)

    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()

    args = TrainingArguments(output_dir="/tmp/arc_trainer", **train_args)
    try:
        return Trainer(model=model, args=args, data_collator=collator,
                       train_dataset=dataset, processing_class=tokenizer,
                       callbacks=callbacks)
    except TypeError:
        return Trainer(model=model, args=args, data_collator=collator,
                       train_dataset=dataset, tokenizer=tokenizer,
                       callbacks=callbacks)


def unwrap(trainer, model):
    try:
        return trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)
    except (AttributeError, TypeError):
        return model


_cache_reported = False


def prepare_cache(cache, pos):
    """Return a KV cache holding exactly `pos` tokens.

    The DFS explores sibling continuations of the same prefix, so every forward
    at a given depth must start from that prefix. Legacy tuple caches are
    immutable — each forward returns a fresh tuple and the parent's cache is
    untouched — but a modern `Cache` object is mutated in place and handed
    back, so without cropping, sibling branches would each extend the previous
    one's cache and the search would silently decode nonsense.
    """
    global _cache_reported
    crop = getattr(cache, "crop", None)
    if not _cache_reported:
        _cache_reported = True
        kind = type(cache).__name__
        print(f"*** kv cache: {kind}, "
              + ("croppable" if crop is not None
                 else "no crop() — assumed immutable per forward"))
    if crop is not None:
        try:
            crop(pos)
        except (AttributeError, TypeError, IndexError):
            pass
    return cache


In [ ]:
%%writefile arc_config.py
"""Compute budget configuration for the two-pass solver.

Pass 1 runs every task on a cheap budget. The confidence of each pass-1 result
decides which tasks are re-run in pass 2 with a richer budget. Setting
BASELINE=True reproduces the public 33.89 notebook exactly (single pass, the
original hyper-parameters) and is kept as a safety net.
"""

import numpy as np

BASELINE = False

PASS1_TIME_FRACTION = 0.55

PASS2_TASK_FRACTION = 0.40

import os

MODEL_OWNER = "sorokin"
MODEL_SLUG = "qwen3_4b_grids15_sft139"

MODEL_PATH = f"/kaggle/input/models/{MODEL_OWNER}/{MODEL_SLUG}/transformers/bfloat16/1"

INPUT_ROOT = "/kaggle/input"


def find_model_dirs(root=INPUT_ROOT, max_depth=5):
    """Directories under `root` that look like a loadable checkpoint."""
    hits = []
    if not os.path.isdir(root):
        return hits
    for dirpath, dirnames, filenames in os.walk(root):
        depth = os.path.relpath(dirpath, root).count(os.sep)
        if os.path.relpath(dirpath, root) == ".":
            depth = 0
        if depth >= max_depth:
            dirnames[:] = []
        if "config.json" in filenames and any(
            f.endswith((".safetensors", ".bin")) for f in filenames
        ):
            hits.append(dirpath)
            dirnames[:] = []
    return sorted(hits)


def resolve_model_path(root=INPUT_ROOT, expected=None):
    """The checkpoint to load, or None if nothing usable is attached.

    Falls back to searching because the version number in MODEL_PATH is the
    part most likely to drift, and a wrong path otherwise surfaces deep inside
    transformers as an unhelpful "Repo id must be in the form ..." error: it
    treats a non-existent directory as a Hugging Face repo name and tries to
    download it.
    """
    expected = expected or MODEL_PATH
    if os.path.isdir(expected):
        return expected

    hits = find_model_dirs(root)
    named = [h for h in hits if MODEL_SLUG in h]
    if named:
        return named[0]
    if len(hits) == 1:
        return hits[0]
    return None

NUM_WORKERS = None


def resolve_num_workers(device_count):
    if NUM_WORKERS is not None:
        return NUM_WORKERS
    return max(1, device_count)

DEV_TASK_IDS = [
    "0934a4d8", "36a08778", "981571dc", "aa4ec2a5",
    "20270e3b",
    "db0c5428",
    "d8e07eb2",
    "13e47133",
]

MAX_SEQ_LENGTH = 8192

LORA_RANK = 256

TTT_TIME_FRACTION = 0.6

TOKENS_PER_SECOND_PRIOR = 950.0

DEV_TIME_BUDGET_SECONDS = 900

GLOBAL_RESERVE_SECONDS = 1800

OUTPUT_DIRS = {
    1: "/kaggle/inference_outputs",
    2: "/kaggle/inference_outputs_p2",
}

RUN_NAMES = {1: "", 2: "_p2"}


class PassBudget:
    """Per-pass knobs. `max_score` is a negative-log-probability cutoff for the
    DFS decoder, so a larger value explores more (and costs more)."""

    def __init__(self, ttt_augmentations, ttt_epochs, eval_permutations,
                 max_score, per_task_cap, dfs_cap, adaptive_cap=True):
        self.ttt_augmentations = ttt_augmentations
        self.ttt_epochs = ttt_epochs
        self.eval_permutations = eval_permutations
        self.max_score = max_score
        self.per_task_cap = per_task_cap
        self.dfs_cap = dfs_cap
        self.adaptive_cap = adaptive_cap


BASELINE_BUDGET = PassBudget(
    ttt_augmentations=16,
    ttt_epochs=1,
    eval_permutations=2,
    max_score=-np.log(0.2),
    per_task_cap=1200,
    dfs_cap=540,
    adaptive_cap=False,
)

PASS_BUDGETS = {
    1: PassBudget(
        ttt_augmentations=16,
        ttt_epochs=1,
        eval_permutations=2,
        max_score=-np.log(0.2),
        per_task_cap=600,
        dfs_cap=300,
    ),
    2: PassBudget(
        ttt_augmentations=24,
        ttt_epochs=2,
        eval_permutations=4,
        max_score=-np.log(0.12),
        per_task_cap=1500,
        dfs_cap=540,
    ),
}

if BASELINE:
    PASS_BUDGETS = {1: BASELINE_BUDGET}
    PASS1_TIME_FRACTION = 1.0
    PASS2_TASK_FRACTION = 0.0


In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score


In [ ]:
%%writefile arc_batching.py
def build_decode_batches(test_id_to_subkeys, n_perm):
    """Batches of four augmentations, ordered so the earliest batches already
    cover diverse transforms.

    `ArcDataset.augment` produces eight dihedral groups (identity/transpose
    crossed with rot90^k), each holding `n_perm` colour permutations, and
    sorting the keys lays those groups out contiguously. A batch pairs two
    groups two rotations apart so it never spends all four slots on
    near-identical views: when a task is cut short by the time budget, what we
    did decode is still spread over the symmetry group. Non-transposed groups
    are scheduled first, and all test outputs get round r before any gets
    round r+1.
    """
    pair_order = [(0, 2), (1, 3), (4, 6), (5, 7)]

    batches = []
    for p in range(0, n_perm, 2):
        for a, b in pair_order:
            for subkeys in test_id_to_subkeys.values():
                batch = (subkeys[a*n_perm + p : a*n_perm + p + 2]
                         + subkeys[b*n_perm + p : b*n_perm + p + 2])
                if batch:
                    batches.append(batch)
    return batches


In [ ]:
%%writefile arc_labels.py
"""Building the training labels for completion-only test-time training.

The reference collator scans for two hard-coded token ids — 11 for `user` and
12 for `assistant` — and supervises the span from two tokens past an assistant
marker to just past the next end-of-turn. That works as long as the tokeniser
lays the chat markers out exactly as it did for the reference. When it does
not, the scan finds nothing, every label stays at -100, and test-time training
silently becomes a no-op: no error, no loss, an adapter that never moves.

So derive the marker token sequences from the tokeniser at run time and search
for those sequences instead. With the expected 16-token vocabulary this
produces exactly the same labels as the reference; when the ids differ it still
produces the right ones.
"""

IGNORE_INDEX = -100


def find_subsequence(ids, seq):
    """Every start position at which `seq` occurs in `ids`."""
    if not seq:
        return []
    n, m = len(ids), len(seq)
    return [i for i in range(n - m + 1) if ids[i:i + m] == seq]


def assistant_starts(ids, assistant_header, user_header=None):
    """Positions of the assistant headers in the transcript.

    This model's vocabulary has no room for the words "user" and "assistant",
    so both headers tokenise to the same thing and cannot be told apart by
    content. They can be told apart by position: the transcript the formatter
    writes strictly alternates user, assistant, user, assistant, so with
    indistinguishable headers every second occurrence, starting from the
    second, is an assistant turn.
    """
    starts = find_subsequence(ids, assistant_header)
    if user_header is not None and list(user_header) == list(assistant_header):
        return starts[1::2]
    return starts


def build_completion_labels(ids, assistant_header, end_marker,
                            user_header=None, ignore_index=IGNORE_INDEX):
    """Supervise each assistant turn: its content plus the closing marker.

    Everything else — the prompt, the demonstration inputs, the headers
    themselves and the padding — is ignored. Supervising the inputs too would
    spend half the gradient teaching the model to copy grids it was given.
    An assistant header with no end marker after it is skipped rather than run
    to the end of the sequence.
    """
    ids = list(ids)
    labels = [ignore_index] * len(ids)

    ends = find_subsequence(ids, end_marker)
    for pos in assistant_starts(ids, assistant_header, user_header):
        start = pos + len(assistant_header)
        end = next((e for e in ends if e >= start), None)
        if end is None:
            continue
        stop = end + len(end_marker)
        labels[start:stop] = ids[start:stop]

    return labels


def marker_sequences(tokenizer):
    """Token ids for the chat markers the formatter writes."""
    def encode(text):
        try:
            return list(tokenizer.encode(text, add_special_tokens=False))
        except TypeError:
            return list(tokenizer.encode(text))

    return {
        "user": encode("<|im_start|>user\n"),
        "assistant": encode("<|im_start|>assistant\n"),
        "end": encode("<|im_end|>"),
    }


In [ ]:
%%writefile arc_confidence.py
"""Confidence scoring of pass-1 results, used to pick the pass-2 queue.

Kept free of torch/unsloth imports so it can be unit tested off-GPU.
"""

import os
import bz2
import pickle
import numpy as np


def hashable(guess):
    return tuple(map(tuple, guess))


def _group_by_solution(guesses):
    groups = {}
    for g in guesses:
        groups.setdefault(hashable(g["solution"]), []).append(g)
    return groups


def getter_kgmon(guesses):
    """Baseline ranking score: how many decode results produced this exact grid,
    minus the mean augmentation NLL of those results (lower NLL is better)."""
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score


def output_confidence(guesses, expected_results):
    """Confidence in [0, 1] for one test output.

    Three signals, all cheap and already present in the decode results:

    - coverage: did the decoder produce anything at all? A task where the DFS
      returned two beams out of sixteen attempts is not "confident", it is
      starved, and starved tasks are exactly what pass 2 should revisit.
    - support: what share of the decode results agree on the top grid.
    - margin: how far the top grid outscores the runner-up. Undefined when
      everything agreed, which is the good case, so it is treated as maximal.
    """
    if not guesses:
        return 0.0

    groups = _group_by_solution(guesses)
    scored = sorted((getter_kgmon(g) for g in groups.values()), reverse=True)

    n_results = len(guesses)
    coverage = min(1.0, n_results / max(1, expected_results))
    support = max(len(g) for g in groups.values()) / n_results

    if len(scored) > 1:
        margin = float(np.clip((scored[0] - scored[1]) / 4.0, 0.0, 1.0))
    else:
        margin = 1.0

    return coverage * (0.6 * support + 0.4 * margin)


def load_pass_results(store):
    """subkey-store -> {base_key: [guess, ...]}, base_key being `{task}_{index}`."""
    results = {}
    if not os.path.isdir(store):
        return results
    for name in os.listdir(store):
        try:
            with bz2.BZ2File(os.path.join(store, name)) as f:
                outputs = pickle.load(f)
        except Exception as e:
            print(f"*** Skipping unreadable result '{name}': {e}")
            continue
        results.setdefault(name.split(".")[0], []).extend(outputs)
    return results


def rank_tasks(task_ids, store, expected_results):
    """Rank every task from least to most confident.

    Tasks pass 1 never reached have no results and sort first, ahead of every
    task that produced something. A task with several test outputs is only as
    confident as its weakest output.
    """
    per_output = load_pass_results(store)

    by_task = {}
    for base_key, guesses in per_output.items():
        task_id = base_key.rsplit("_", 1)[0]
        conf = output_confidence(guesses, expected_results)
        by_task[task_id] = min(conf, by_task.get(task_id, 1.0))

    ranked = [(by_task.get(t, -1.0), t) for t in task_ids]
    ranked.sort(key=lambda x: (x[0], x[1]))
    return ranked


def select_pass2_queue(task_ids, store, expected_results, fraction):
    """Least-confident `fraction` of the tasks, plus every unreached task."""
    ranked = rank_tasks(task_ids, store, expected_results)

    unreached = [t for conf, t in ranked if conf < 0.0]
    reached = [t for conf, t in ranked if conf >= 0.0]

    n_take = int(round(fraction * len(reached)))
    return unreached + reached[:n_take]


In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


def infer_shape_rule(train_pairs):
    """Rule that maps an input shape to the expected output shape, or None."""
    if len(train_pairs) < 2:
        return None

    ins = [np.shape(p["input"]) for p in train_pairs]
    outs = [np.shape(p["output"]) for p in train_pairs]
    if any(len(s) != 2 for s in ins + outs):
        return None

    if all(i == o for i, o in zip(ins, outs)):
        return ("same", None)

    if len(set(outs)) == 1:
        return ("const", outs[0])

    ratios = set()
    for (ih, iw), (oh, ow) in zip(ins, outs):
        if ih and iw and oh % ih == 0 and ow % iw == 0:
            ratios.add((oh // ih, ow // iw))
        else:
            ratios = None
            break
    if ratios and len(ratios) == 1:
        return ("scale", ratios.pop())

    ratios = set()
    for (ih, iw), (oh, ow) in zip(ins, outs):
        if oh and ow and ih % oh == 0 and iw % ow == 0:
            ratios.add((ih // oh, iw // ow))
        else:
            ratios = None
            break
    if ratios and len(ratios) == 1:
        return ("div", ratios.pop())

    return None


def expected_shape(rule, input_shape):
    if rule is None:
        return None
    kind, param = rule
    ih, iw = input_shape
    if kind == "same":
        return (ih, iw)
    if kind == "const":
        return param
    if kind == "scale":
        return (ih * param[0], iw * param[1])
    if kind == "div":
        if param[0] and param[1] and ih % param[0] == 0 and iw % param[1] == 0:
            return (ih // param[0], iw // param[1])
    return None


def apply_shape_prior(ordered_outputs, query):
    """Stable partition of the ranked candidates: conforming grids first."""
    if not ordered_outputs or query is None:
        return ordered_outputs

    rule = infer_shape_rule(query.get("train", []))
    target = expected_shape(rule, np.shape(query["test"][0]["input"]))
    if target is None:
        return ordered_outputs

    conforming = [g for g in ordered_outputs if np.shape(g) == target]
    if not conforming or len(conforming) == len(ordered_outputs):
        return ordered_outputs

    rest = [g for g in ordered_outputs if np.shape(g) != target]
    return conforming + rest


class ArcDecoder:

    def __init__(self, dataset, n_guesses, use_shape_prior=True):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.use_shape_prior = use_shape_prior
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        if not os.path.isdir(store):
            print(f"*** No results at '{store}', skipping.")
            return self
        n_loaded = 0
        for key in os.listdir(store):
            try:
                with bz2.BZ2File(os.path.join(store, key)) as f:
                    outputs = pickle.load(f)
            except Exception as e:
                print(f"*** Skipping unreadable result '{key}': {e}")
                continue
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample
                n_loaded += 1
        print(f"*** Loaded {n_loaded} results from '{store}' for {len(self.decoded_results)} outputs.")
        return self

    def run_selection_algo(self, selection_algorithm=score_kgmon, use_shape_prior=None):
        if use_shape_prior is None:
            use_shape_prior = self.use_shape_prior
        selected = {}
        for bk, v in self.decoded_results.items():
            ordered = selection_algorithm({k: g for k, g in v.items()})
            if use_shape_prior:
                ordered = apply_shape_prior(ordered, self.dataset.queries.get(bk))
            selected[bk] = ordered
        return selected

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        if correct_beam_scores:
            print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
            print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            for use_prior in [False, True]:
                name = selection_algorithm.__name__ + (" + shape prior" if use_prior else "")
                selected = self.run_selection_algo(selection_algorithm, use_shape_prior=use_prior)
                correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
                score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
                print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")


In [ ]:
%%writefile arc_solver.py
import arc_backend
from arc_loader import ArcDataset, QwenFormatter
import arc_config
from arc_batching import build_decode_batches
from arc_labels import build_completion_labels, marker_sequences

if arc_backend.HAS_UNSLOTH:
    from unsloth import UnslothTrainingArguments, UnslothTrainer
else:
    UnslothTrainer = object

import gc
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling, TrainerCallback

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):


    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        if hasattr(loss, "clone"):
            loss = loss.clone()
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class DeadlineCallback(TrainerCallback):
    """Stop test-time training when its share of the task budget is spent.

    Training a long task to completion can consume the whole per-task cap, and
    the decoder then never runs: the task costs full price and returns nothing.
    Cutting training short leaves a less adapted model but one that still gets
    to answer.
    """

    def __init__(self, deadline):
        self.deadline = deadline
        self.stopped_early = False

    def on_step_end(self, args, state, control, **kwargs):
        if time.time() > self.deadline:
            self.stopped_early = True
            control.should_training_stop = True
        return control


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    assistant_header = None
    user_header = None
    end_marker = None

    supervised = None
    total = None

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = build_completion_labels(
                batch["input_ids"][i].tolist(),
                self.assistant_header,
                self.end_marker,
                user_header=self.user_header,
            )
            batch["labels"][i] = torch.tensor(
                labels, dtype=batch["labels"].dtype, device=batch["labels"].device)

        if QwenDataCollatorForCompletionOnlyLM.supervised is None:
            QwenDataCollatorForCompletionOnlyLM.supervised = int((batch["labels"] != -100).sum())
            QwenDataCollatorForCompletionOnlyLM.total = int(batch["labels"].numel())
        return batch


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time, dfs_cap=540) -> dict:

    n = logits.size(0)

    nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for t in ARC_TOKENS:
            score = nll[i, t].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0])
    
    while time.time() - start_time < dfs_cap and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=arc_backend.prepare_cache(cache, pos),
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
            dfs_cap=dfs_cap,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time, dfs_cap=540):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    if outputs.past_key_values is None:
        raise RuntimeError(
            "model returned no KV cache; decoding needs use_cache to be honoured"
        )
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
        dfs_cap=dfs_cap,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    batch_logits = outputs.logits.float().cpu().log_softmax(-1)
    result = []
    for logits, query_tokens, answer_tokens in zip(batch_logits, batch_query_tokens, batch_answer_tokens):
        query_length = len(query_tokens)
        answer_logits = logits[query_length-1:query_length-1+len(answer_tokens)]
        answer_score = answer_logits[torch.arange(len(answer_tokens)), answer_tokens].sum()
        result.append(-answer_score.item())
    return result


def worker(rank, queue, end_time, pass_id=1, n_workers=1):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    budget = arc_config.PASS_BUDGETS[pass_id]

    supports_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    compute_dtype = torch.bfloat16 if supports_bf16 else torch.float16
    print(f"[Rank {rank}] compute dtype: {compute_dtype}")


    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=budget.ttt_epochs,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        logging_strategy="no",
        fp16=not supports_bf16,
        bf16=supports_bf16,
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    train_args[arc_backend.eval_strategy_key()] = "no"

    max_seq_length = arc_config.MAX_SEQ_LENGTH

    model_path = arc_config.resolve_model_path()
    if model_path is None:
        raise FileNotFoundError(
            f"No model checkpoint under {arc_config.INPUT_ROOT}. Expected "
            f"{arc_config.MODEL_PATH}; attach "
            f"{arc_config.MODEL_OWNER}/{arc_config.MODEL_SLUG}."
        )
    print(f"[Rank {rank}] model: {model_path}")

    model, tokenizer = arc_backend.load_model(
        model_path, max_seq_length, compute_dtype, PAD_ID)

    model = arc_backend.get_peft_model(model)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(compute_dtype)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    markers = marker_sequences(tokenizer)
    collator.assistant_header = markers["assistant"]
    collator.user_header = markers["user"]
    collator.end_marker = markers["end"]
    print(f"[Rank {rank}] markers: user={markers['user']} "
          f"assistant={markers['assistant']} end={markers['end']}")
    if not markers["assistant"] or not markers["end"]:
        raise RuntimeError(
            "tokenizer does not produce the chat markers the formatter writes; "
            "test-time training would have nothing to supervise"
        )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = budget.max_score

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    tokens_per_second = arc_config.TOKENS_PER_SECOND_PRIOR
    reported = False

    dir_outputs = arc_config.OUTPUT_DIRS[pass_id]
    os.makedirs(dir_outputs, exist_ok=True)

    while not queue.empty():

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break

        start_time = time.time()

        if budget.adaptive_cap:
            try:
                remaining_tasks = max(1.0, queue.qsize() / n_workers + 1)
            except NotImplementedError:
                remaining_tasks = 1.0
            task_cap = min(budget.per_task_cap,
                           max(120.0, (end_time - start_time) / remaining_tasks))
        else:
            task_cap = budget.per_task_cap
        
        try:
            torch.cuda.reset_peak_memory_stats()

            load_result = set_peft_model_state_dict(
                model,
                default_weights.copy(),
                adapter_name="default",
            )

            model = arc_backend.for_training(model)

            deadline = DeadlineCallback(min(
                start_time + task_cap * arc_config.TTT_TIME_FRACTION,
                end_time,
            ))

            puzzle_ds = arc_test_set.change_keys([key])

            train_ds = puzzle_ds.augment(n=budget.ttt_augmentations, shfl_keys=True, seed=1)
            train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)
            samples = train_ds.as_list(formatter)

            task_tokens = max(1, len(tokenizer.encode(samples[0]["text"])))
            step_seconds = task_tokens / tokens_per_second

            task_train_args = dict(train_args)
            affordable = int(task_cap * arc_config.TTT_TIME_FRACTION / step_seconds)
            planned = int(np.clip(affordable, 8, len(samples) * budget.ttt_epochs))
            task_train_args["max_steps"] = planned
            task_train_args.pop("num_train_epochs", None)

            with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
                trainer = arc_backend.build_trainer(
                    model=model,
                    tokenizer=tokenizer,
                    collator=collator,
                    samples=samples,
                    train_args=task_train_args,
                    max_seq_length=max_seq_length,
                    fixed_trainer_cls=UnslothFixedTrainer,
                    callbacks=[deadline],
                )

                stats = trainer.train()

                model = arc_backend.unwrap(trainer, model)

                del trainer

            model = arc_backend.for_inference(model)
        
            gc.collect()
            torch.cuda.empty_cache()
            
            memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
            print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

            torch.cuda.reset_peak_memory_stats()
        
            if stats.global_step and stats.metrics.get("train_runtime"):
                measured = task_tokens * stats.global_step / stats.metrics["train_runtime"]
                tokens_per_second = 0.7 * tokens_per_second + 0.3 * measured
                print(f"[Rank {rank}] planned {planned} steps, ran "
                      f"{stats.global_step}; throughput {measured:.0f} tok/s")

            if QwenDataCollatorForCompletionOnlyLM.supervised is not None and not reported:
                reported = True
                sup = QwenDataCollatorForCompletionOnlyLM.supervised
                tot = QwenDataCollatorForCompletionOnlyLM.total
                print(f"[Rank {rank}] collator: {sup}/{tot} supervised label positions")
                if sup == 0:
                    print(f"[Rank {rank}] WARNING: nothing supervised — "
                          f"test-time training cannot learn")
                current = get_peft_model_state_dict(model, adapter_name="default")
                delta = max(
                    (current[k].float() - default_weights[k].float()).norm().item()
                    for k in list(current)[:8]
                )
                print(f"[Rank {rank}] adapter delta after training: {delta:.3e}")
                if delta == 0.0:
                    print(f"[Rank {rank}] WARNING: adapter unchanged — "
                          f"test-time training is a no-op")

            if deadline.stopped_early:
                print(f"[Rank {rank}] training hit its deadline")
            print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

            puzzle_ds_multi = puzzle_ds.split_multi_replies()

            eval_ds = puzzle_ds_multi.augment(n=budget.eval_permutations, seed=2)
            eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

            test_id_to_subkeys = defaultdict(list)
            for subkey in sorted(eval_ds.keys):
                test_id = subkey.split(".")[0].split("_")[1]
                test_id_to_subkeys[test_id].append(subkey)

            batches = build_decode_batches(test_id_to_subkeys, budget.eval_permutations)

            with torch.inference_mode():
                
                known_scores = {}

                for subkeys in batches:

                    spend_time = time.time() - start_time
                    if spend_time > task_cap or time.time() > end_time:
                        print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                        break

                    print(f"[Rank {rank}] decoding {subkeys}")

                    tokens = []
                    for subkey in subkeys:
                        data = eval_ds.get(subkey, formatter)
                        tokens.append(tokenizer.encode(data["input"]))

                    dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time, dfs_cap=budget.dfs_cap)

                    for subkey_id, scored_beams in dfs_result:

                        subkey = subkeys[subkey_id]
                        bk = subkey.split(".")[0]
                        decoded_result = []

                        for beam_score, tokens in scored_beams:

                            array = formatter.convert_tokens_to_array(tokens)
                            if array is None:
                                continue

                            solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                            grid_id = (bk, tuple(map(tuple, solution)))

                            if grid_id in known_scores:
                                augmented_scores = known_scores[grid_id]
                            else:
                                print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                                aug_dataset = ArcDataset(
                                    keys=[bk],
                                    queries={bk: puzzle_ds_multi.queries.get(bk)},
                                    replies={bk: [solution.tolist()]},
                                )
                                aug_dataset = aug_dataset.augment(seed=hash(bk) % 1024**2)
                                aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                                aug_queries = []
                                aug_answers = []
                                for augmented_sample in aug_dataset.as_list(formatter):
                                    aug_queries.append(augmented_sample["input"])
                                    aug_answers.append(augmented_sample["reply"])
                                augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                                augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                                augmented_scores = augmented_scores1 + augmented_scores2
                                known_scores[grid_id] = augmented_scores
                        
                            decoded_result.append({
                                "beam_score": beam_score,
                                "score_aug": augmented_scores,
                                "solution": solution,
                            })

                        if len(decoded_result):
                            with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                                pickle.dump(decoded_result, f)

            memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
            print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
            spend_time = time.time() - start_time
            print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")

        except torch.OutOfMemoryError:
            print(f"[Rank {rank}] OUT OF MEMORY on {key}, skipping it")
        except Exception as e:
            print(f"[Rank {rank}] FAILED on {key}: {type(e).__name__}: {e}")
        finally:
            gc.collect()
            torch.cuda.empty_cache()


In [ ]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import torch.multiprocessing as mp

import arc_config
from arc_confidence import select_pass2_queue


def local_worker(rank, queue, end_time, pass_id, n_workers):

    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    if rank > 0:
        while not os.path.exists(f"/kaggle/worker{rank-1}_p{pass_id}"):
            time.sleep(5)

    try:
        import unsloth
    except ImportError:
        pass

    from arc_solver import worker

    with open(f"/kaggle/worker{rank}_p{pass_id}", "w") as f:
        f.write("Ok")

    print(f"[Rank {rank}] start pass {pass_id}!")

    worker(rank, queue, end_time, pass_id=pass_id, n_workers=n_workers)

    print(f"[Rank {rank}] done pass {pass_id}!")


def run_pass(pass_id, task_ids, end_time, n_workers):
    queue = mp.Manager().Queue()
    for key in task_ids:
        queue.put(key)
    for _ in range(n_workers):
        queue.put(None)

    print(f"*** Pass {pass_id}: {len(task_ids)} tasks, {n_workers} workers, "
          f"{(end_time - time.time())/3600:.2f}h budget.", flush=True)

    mp.spawn(local_worker, args=(queue, end_time, pass_id, n_workers),
             nprocs=n_workers)


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    task_ids = sorted(data.keys())
    if not rerun_mode:
        task_ids = [k for k in task_ids if k in arc_config.DEV_TASK_IDS]

    n_workers = arc_config.resolve_num_workers(torch.cuda.device_count())
    print(f"*** {torch.cuda.device_count()} GPU(s) visible, using {n_workers} worker(s).")

    global_end = args.end_time
    if not rerun_mode:
        global_end = min(global_end,
                         time.time() + arc_config.DEV_TIME_BUDGET_SECONDS)
        print(f"*** dev run: {arc_config.DEV_TIME_BUDGET_SECONDS/60:.0f} min budget "
              f"(the rerun gets the full 12h)")

    total_budget = global_end - time.time()
    pass1_end = time.time() + total_budget * arc_config.PASS1_TIME_FRACTION

    run_pass(1, task_ids, pass1_end, n_workers)

    if len(arc_config.PASS_BUDGETS) > 1 and time.time() < global_end:
        expected = 8 * arc_config.PASS_BUDGETS[1].eval_permutations
        pass2_ids = select_pass2_queue(
            task_ids,
            arc_config.OUTPUT_DIRS[1],
            expected_results=expected,
            fraction=arc_config.PASS2_TASK_FRACTION,
        )
        if pass2_ids:
            run_pass(2, pass2_ids, global_end, n_workers)
        else:
            print("*** Pass 2: nothing to revisit.")
    else:
        print("*** Pass 2 skipped (baseline mode or out of time).")


In [ ]:
import preflight
import arc_config

preflight.check([
    "/kaggle/input/competitions/arc-prize-2026-arc-agi-2",
])


In [ ]:
import os
import subprocess
import sys

env = dict(
    os.environ,
    UNSLOTH_DISABLE_STATISTICS="1",
    TRITON_PTXAS_PATH="/usr/local/cuda/bin/ptxas",
    OMP_NUM_THREADS="12",
    PYTORCH_ALLOC_CONF="expandable_segments:True",
    PYTHONUNBUFFERED="1",
)

proc = subprocess.Popen(
    [sys.executable, "starter.py", "--end-time", str(global_end_time)],
    env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end="")
starter_exit = proc.wait()
print(f"*** starter.py exit code: {starter_exit}")
if starter_exit != 0:
    raise RuntimeError(
        f"starter.py failed (exit {starter_exit}). Stopping here rather "
        f"than writing a submission full of placeholders."
    )


In [ ]:
import os
import json
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder
import arc_config

rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

if rerun_mode:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json")
else:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
    data = data.load_replies("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json")

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)

for pass_id, store in sorted(arc_config.OUTPUT_DIRS.items()):
    decoder.load_decoded_results(store, run_name=arc_config.RUN_NAMES[pass_id])

n_results = sum(len(v) for v in decoder.decoded_results.values())
print(f"*** {n_results} decode results over {len(decoder.decoded_results)} outputs")
if rerun_mode and n_results == 0:
    raise RuntimeError(
        "no decode results at all: every prediction would be a placeholder and "
        "the submission would score zero. Check the solver output above."
    )

submission = data.get_submission(decoder.run_selection_algo())

with open("submission.json", "w") as f:
    json.dump(submission, f)

assert set(submission.keys()) == set(data.keys), "submission is missing task ids"
for k, outputs in submission.items():
    assert len(outputs) == len(data.queries[k]["test"]), f"{k}: wrong number of outputs"
    for o in outputs:
        assert "attempt_1" in o and "attempt_2" in o, f"{k}: missing attempt"
print(f"*** submission.json written for {len(submission)} tasks.")

if not rerun_mode:
    decoder.benchmark_selection_algos()
    with open("submission.json", "r") as f:
        reload_submission = json.load(f)
    print("*** Reload score (contaminated, dev signal only):", data.validate_submission(reload_submission))
